# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. The analysis follows the dataset's Croissant schema, ensuring all record sets, fields, and columns are referenced by their `@id` identifiers for reproducibility.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The Croissant schema enables programmatic access to the description, fields, and data structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

warnings.filterwarnings('ignore')
# Define the Croissant schema URL for FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url=croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"License: {metadata.license}\nPublished: {metadata.datePublished}")

## 2. Data Overview
List the available record sets and their IDs using the Croissant schema. For each record set, examine its available fields and their `@id` attributes.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.record_sets

if len(record_sets) == 0:
    print('No record sets defined in the dataset metadata.')
else:
    for rs in record_sets:
        print(f"Record Set: {rs.id}")
        print(f"  Name: {rs.name}")
        print(f"  Description: {rs.description}")
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    Field: {field.id} (type: {field.data_type})")
        print('-'*40)

# Preview records for the first record set (if available)
if len(record_sets) > 0:
    first_rs_id = record_sets[0].id
    print(f"\nFirst 2 records from Record Set {first_rs_id}:")
    for i, record in enumerate(dataset.records(record_set=first_rs_id)):
        if i > 1: break
        print(record)

## 3. Data Extraction
Load tabular data from each record set into a pandas DataFrame for easy manipulation and analysis, referencing record set and field `@id` values dynamically.

In [ ]:
# Extract all records for each record set
dfs = {}
for rs in record_sets:
    rs_id = rs.id
    records = list(dataset.records(record_set=rs_id))
    if records:
        dfs[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for Record Set: {rs_id}")
    else:
        print(f"No records found for Record Set: {rs_id}")

# Show columns for first loaded DataFrame
if dfs:
    first_rs_id = next(iter(dfs.keys()))
    print(f"\nColumns in DataFrame for Record Set {first_rs_id}:")
    print(dfs[first_rs_id].columns.tolist())
    dfs[first_rs_id].head()
else:
    print('No tabular records could be loaded.')

## 4. Exploratory Data Analysis (EDA)
Now apply example data processing steps. Filter on a numeric field, normalize, and group by a categorical field. All field references use their `@id` values.

In [ ]:
# Choose the primary DataFrame for EDA
if dfs:
    record_set_id = first_rs_id
    df = dfs[record_set_id]

    # List field @ids and DTypes for selection
    print('Available fields and their data types:')
    for col in df.columns:
        print(f"  {col} | dtype: {df[col].dtype}")

    # Attempt to infer a numeric field and a group/categorical field
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]

    # Use first numeric field found, and a string/categorical one if present
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        print('No numeric fields detected for filtering.')
        numeric_field_id = None

    if group_fields:
        group_field_id = group_fields[0]
    else:
        group_field_id = None

    if numeric_field_id is not None:
        # Example threshold: use mean as a cutoff
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        field_norm = f"{numeric_field_id}_normalized"
        filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, field_norm]].head())

        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped filtered data by {group_field_id} (mean of numeric fields):")
            print(grouped_df.head())
    else:
        print('No numeric field available for EDA.')
else:
    print('No records available for EDA.')

## 5. Visualization
Visualize the distribution of the selected numeric field and compare across the grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and numeric_field_id is not None:
    # Distribution plot
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group if grouping field available
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded metadata and tabular data from the FAIR^2 dataset using the Croissant schema and `mlcroissant`. By referencing all fields and entities using their `@id` values, we ensured the reproducibility and clarity of data exploration.

- **Summary:**
    - Dataset covers socio-demographics, gender roles, and knowledge adoption among northern Kenya pastoralist households.
    - Tabular outcomes (e.g., regression coefficients, log likelihoods) were loaded and explored.
    - Numeric field distributions and grouping by categorical features were visualized.

Further analysis can include advanced modeling of adoption predictors, exploring missing values, and more granular hypothesis testing using the provided schema-guided structure.